# Estanque Cilíndrico — Análisis Sísmico según API650 y NCh2369:2025

**Versión mejorada con visualizaciones profesionales**

In [ ]:
import calctext.render
from calctext import render
from math import pi, sqrt, sin, cos, tan, atan, tanh, cosh, sinh
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyBboxPatch
import matplotlib.patches as mpatches
import unitcalc
unitcalc.environment('structural', top_level=True)

# Forzar la unidad de visualización por dimensión
unitcalc.environment.set_base_unit('force',    'kN')
unitcalc.environment.set_base_unit('pressure', 'MPa')
unitcalc.environment.set_base_unit('length',   'mm')
unitcalc.environment.set_base_unit('mass',     'kg')

# Configuración para gráficos export-ready
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 10
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['legend.fontsize'] = 9
plt.rcParams['figure.titlesize'] = 13

# 1. Parámetros de Entrada

In [ ]:
%%render params
# Geometría
r_tanque = 6 * m
e_pared = 5 * mm
h_pared = 10.5 * m
gamma_acero = 78.5 * kN / m**3
gamma_agua = 9.81 * kN / m**3
g = 9.81 * m / s**2
e_fondo = 10 * mm
h_liq = 8.84 * m
h_total = 10.5 * m

# 2. Cálculo de Masas

In [ ]:
%%render
PP_pared = (2*pi*r_tanque) * e_pared * h_pared * gamma_acero
M_pared = PP_pared / g

PP_fondo = pi * r_tanque**2 * e_fondo * gamma_acero
M_fondo = PP_fondo / g

V_liq = pi * r_tanque**2 * h_liq
PP_liq = V_liq * gamma_agua
M_liq = PP_liq / g

PP_techo = 50 * kN
M_techo = PP_techo / g

# 3. Modelo de Masas Impulsiva y Convectiva

In [ ]:
%%render
D = 2 * r_tanque
rel = h_liq / D

In [ ]:
%%render
m_i = M_liq * (tanh(0.866 * D / h_liq) / (0.866 * D / h_liq))

In [ ]:
%%render
m_c = M_liq*(0.23*tanh(3.68*h_liq/D)/(h_liq/D))

### Alturas de aplicación de presión

In [ ]:
%%render
h_i = h_liq*(0.375 if h_liq/D <= 0.75 else (0.5-0.09375/(h_liq/D)))

In [ ]:
%%render
h_c = h_liq*(1-(cosh(3.68*h_liq/D)-1)/(3.68*h_liq/D*sinh(3.68*h_liq/D)))

### Alturas para momento volcante

In [ ]:
%%render
h_prime_i = h_liq * ((0.866*D/h_liq)/(2*tanh(0.866*D/h_liq)) - 0.125 if h_liq/D <= 1.33 else 0.45)

In [ ]:
%%render
h_prime_c = h_liq*(1-(cosh(3.68*h_liq/D)-2.01)/(3.68*h_liq/D*sinh(3.68*h_liq/D)))

### Porcentaje de masa impulsiva y convectiva

In [ ]:
%%render
masa_impulsiva = tanh(0.866*D/h_liq)/(0.866*D/h_liq)
masa_convectiva = 0.23*tanh(3.68*h_liq/D)/(h_liq/D)

# 4. Análisis Sísmico — Periodos Fundamentales

In [ ]:
%%render
E = 200 * GPa

In [ ]:
%%render
C_i = 1/(sqrt(h_liq/D)*(0.46-0.3*h_liq/D+0.067*(h_liq/D)**2))
T_i = (C_i*h_liq*sqrt(gamma_agua/g))/(sqrt(e_pared/D)*sqrt(E)*1000) * s / m

In [ ]:
%%render
C_c = (2*pi)/sqrt(3.68*tanh(3.68*h_liq/D))
T_c = C_c*sqrt(D/g) *s

# 5. Parámetros Sísmicos NCh2369:2025

In [ ]:
%%render
I = 1
R_i = 4
xi_i = 2/100
R_c = 1.5
xi_c = 0.5/100
A_r = 0.42*g

S = 1
r = 4.5
T_0 = 0.3 * s
p = 1.6
q = 3.00
T_1 = 0.27 * s

### Aceleración espectral impulsiva

In [ ]:
%%render
S_ai = (2.75*(I*A_r*S)/(R_i+1)*(0.05/xi_i)**0.4)*1/g

### Aceleración espectral convectiva

In [ ]:
%%render
C_r = 0.16 * R_c
check = C_r * T_1
R_star = 1 if R_c == 1 else (R_c if (T_c) >= (C_r * T_1) else 1.5 + (R_c - 1.5) * (T_c) / (C_r * T_1))
S_ac = (A_r * S * ((1+r*(T_c/T_0)**p)/(1+(T_c/T_0)**q)))*((I/R_star)*1.5)*1/g

# 6. Fuerzas y Momentos Sísmicos

## 6.1 Corte Basal

In [ ]:
%%render
V_i = S_ai*(m_i+M_pared+M_techo)*g
V_c = S_ac*m_c * g
V = sqrt(V_i**2 + V_c**2)*kN
check_V = V / ((m_i+m_c)*g)*1000

## 6.2 Momento Fondo Muro

In [ ]:
%%render
M_i = S_ai*(m_i*h_i+M_pared*h_pared/2+M_techo*h_total)*g
M_c = S_ac*(m_c*h_c)*g
M = sqrt(M_i**2 + M_c**2)*kN*m

## 6.3 Momento Volcante

In [ ]:
%%render
M_prime_i = S_ai*(m_i*h_prime_i+M_pared*h_pared/2+M_techo*h_total+M_fondo*e_fondo/2)*g
M_prime_c = S_ac*(m_c*h_prime_c)*g
M_prime = sqrt(M_prime_i**2 + M_prime_c**2)*kN*m

# 7. Presiones Hidrodinámicas

In [ ]:
# Parámetros para cálculo de presiones
h_liq_num = 8.84
D_num = 12

## 7.1 Presión en Muros - Impulsivo

In [ ]:
def Q_iw(y, h_liq, D):
    return 0.866 * (1 - (y / h_liq)**2) * np.tanh(0.866 * D / h_liq)

# Calcular Q_iw en la base
Q_iw_max = Q_iw(0, h_liq_num, D_num)

In [ ]:
%%render
Q_iw_val = Q_iw_max
p_iw = Q_iw_val * gamma_agua * h_liq * S_ai

## 7.2 Presión en Fondo - Impulsivo

In [ ]:
def Q_ib(x, h_liq, D):
    return 0.866 * (np.sinh(0.866*2*x/h_liq))/(np.cosh(0.866*D/h_liq))

# Calcular Q_ib en borde
Q_ib_max = Q_ib(D_num/2, h_liq_num, D_num)

In [ ]:
%%render
Q_ib_val = Q_ib_max
p_ib = Q_ib_val * gamma_agua * h_liq * S_ai

## 7.3 Presión en Muros - Convectivo

In [ ]:
def Q_cw(y, h_liq, D):
    return 0.5625*(np.cosh(3.674*y/D))/(np.cosh(3.674*h_liq/D))

# Calcular Q_cw en superficie
Q_cw_max = Q_cw(h_liq_num, h_liq_num, D_num)

In [ ]:
%%render
Q_cw_val = Q_cw_max
p_cw = Q_cw_val * gamma_agua * h_liq * S_ac

## 7.4 Presión en Fondo - Convectivo

In [ ]:
def Q_cb(x, h_liq, D):
    return 1.125*((x/D)-(4/3)*(x/D)**3) / np.cosh(3.674*h_liq/D)

# Calcular Q_cb en borde
Q_cb_max = Q_cb(D_num/2, h_liq_num, D_num)

In [ ]:
%%render
Q_cb_val = Q_cb_max
p_cb = Q_cb_val * gamma_agua * h_liq * S_ac

## 7.5 Presión Lineal Equivalente

### Impulsiva

In [ ]:
%%render
q_i = (S_ai*m_i*g)/(pi * D * m / 2)
a_i = (q_i)/((h_liq*m)**2) * (4*h_liq - 6*h_i)
b_i = (q_i)/((h_liq*m)**2) * (6*h_i - 2*h_liq)

### Convectiva

In [ ]:
%%render
q_c = (S_ac*m_c*g)/(pi * D * m / 2)
a_c = (q_c)/((h_liq*m)**2) * (4*h_liq - 6*h_c)
b_c = (q_c)/((h_liq*m)**2) * (6*h_c - 2*h_liq)

## 7.6 Presión por Inercia de Muro

In [ ]:
%%render
p_ww = S_ai * e_pared * gamma_acero

## 7.7 Presión por Acción Sísmica Vertical

In [ ]:
%%render
R_v = 2
T_v = 0.3*s
xi_v = 2/100

S_av = ((I)/(R_v)*(0.05/xi_v)**0.4)*(0.7*A_r*S*((1+r*(1.7*T_v/T_0)**p)/(1+(1.7*T_v/T_0)**q)))*1/g

p_v = S_av * gamma_agua * h_liq

## 7.8 Presión Hidrodinámica Máxima Combinada (SRSS)

In [ ]:
%%render
p = sqrt((p_iw + p_ww)**2 + p_cw**2 + p_v**2) * kN / m**2
p_estatico = (gamma_agua * h_liq)
rel_presion = p / p_estatico

# 8. Altura de Ola y Anclaje

In [ ]:
%%render
d_max = S_ac * 2.5 * D *m / 2

In [ ]:
# Verificación de requerimiento de anclaje
check_anclaje = ("SÍ REQUIERE" if h_liq_num/D_num >= 1/float(S_ai) else "NO REQUIERE")
print(f"¿Requiere anclaje?: {check_anclaje}")
print(f"Relación h/D = {h_liq_num/D_num:.3f}")
print(f"Criterio: h/D ≥ 1/S_ai = {1/float(S_ai):.3f}")

---
# 9. VISUALIZACIONES Y ANÁLISIS DE RESULTADOS

## 9.1 Distribuciones de Presión Hidrodinámica (Mejoradas)

In [ ]:
# Crear figura con 4 subplots para presiones
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Distribuciones de Presión Hidrodinámica', fontsize=14, fontweight='bold')

# 1. Presión en Muro - Impulsivo
y_wall = np.linspace(0, h_liq_num, 100)
Q_iw_dist = Q_iw(y_wall, h_liq_num, D_num)
p_iw_dist = Q_iw_dist * float(gamma_agua) * h_liq_num * float(S_ai)  # en kPa

axes[0,0].plot(p_iw_dist, y_wall, 'b-', linewidth=2.5, label='Impulsivo')
axes[0,0].axhline(y=float(h_i), color='b', linestyle='--', alpha=0.6, linewidth=1.5, label=f'h_i = {float(h_i):.2f} m')
axes[0,0].scatter([p_iw_dist[0]], [0], color='red', s=100, zorder=5, label=f'p_max = {p_iw_dist[0]:.1f} kPa')
axes[0,0].set_xlabel('Presión (kPa)', fontweight='bold')
axes[0,0].set_ylabel('Altura desde fondo (m)', fontweight='bold')
axes[0,0].set_title('Presión en Muro — Impulsivo', fontweight='bold')
axes[0,0].grid(True, alpha=0.3, linestyle=':')
axes[0,0].legend(loc='best')
axes[0,0].set_ylim([0, h_liq_num*1.05])
axes[0,0].set_xlim([0, max(p_iw_dist)*1.1])

# 2. Presión en Muro - Convectivo
Q_cw_dist = Q_cw(y_wall, h_liq_num, D_num)
p_cw_dist = Q_cw_dist * float(gamma_agua) * h_liq_num * float(S_ac)  # en kPa

axes[0,1].plot(p_cw_dist, y_wall, 'g-', linewidth=2.5, label='Convectivo')
axes[0,1].axhline(y=float(h_c), color='g', linestyle='--', alpha=0.6, linewidth=1.5, label=f'h_c = {float(h_c):.2f} m')
axes[0,1].scatter([p_cw_dist[-1]], [h_liq_num], color='red', s=100, zorder=5, label=f'p_max = {p_cw_dist[-1]:.1f} kPa')
axes[0,1].set_xlabel('Presión (kPa)', fontweight='bold')
axes[0,1].set_ylabel('Altura desde fondo (m)', fontweight='bold')
axes[0,1].set_title('Presión en Muro — Convectivo', fontweight='bold')
axes[0,1].grid(True, alpha=0.3, linestyle=':')
axes[0,1].legend(loc='best')
axes[0,1].set_ylim([0, h_liq_num*1.05])
axes[0,1].set_xlim([0, max(p_cw_dist)*1.1])

# 3. Presión en Fondo - Impulsivo
x_bottom = np.linspace(-D_num/2, D_num/2, 100)
Q_ib_dist = Q_ib(x_bottom, h_liq_num, D_num)
p_ib_dist = Q_ib_dist * float(gamma_agua) * h_liq_num * float(S_ai)  # en kPa

axes[1,0].plot(x_bottom, p_ib_dist, 'b-', linewidth=2.5, label='Impulsivo')
axes[1,0].axhline(y=0, color='k', linestyle='-', alpha=0.3, linewidth=0.8)
axes[1,0].axvline(x=0, color='k', linestyle='--', alpha=0.3, linewidth=0.8)
idx_max = np.argmax(np.abs(p_ib_dist))
axes[1,0].scatter([x_bottom[idx_max]], [p_ib_dist[idx_max]], color='red', s=100, zorder=5, label=f'p_max = {abs(p_ib_dist[idx_max]):.1f} kPa')
axes[1,0].set_xlabel('Distancia radial desde centro (m)', fontweight='bold')
axes[1,0].set_ylabel('Presión (kPa)', fontweight='bold')
axes[1,0].set_title('Presión en Fondo — Impulsivo', fontweight='bold')
axes[1,0].grid(True, alpha=0.3, linestyle=':')
axes[1,0].legend(loc='best')
axes[1,0].set_xlim([-D_num/2*1.05, D_num/2*1.05])

# 4. Presión en Fondo - Convectivo
Q_cb_dist = Q_cb(x_bottom, h_liq_num, D_num)
p_cb_dist = Q_cb_dist * float(gamma_agua) * h_liq_num * float(S_ac)  # en kPa

axes[1,1].plot(x_bottom, p_cb_dist, 'g-', linewidth=2.5, label='Convectivo')
axes[1,1].axhline(y=0, color='k', linestyle='-', alpha=0.3, linewidth=0.8)
axes[1,1].axvline(x=0, color='k', linestyle='--', alpha=0.3, linewidth=0.8)
idx_max = np.argmax(np.abs(p_cb_dist))
axes[1,1].scatter([x_bottom[idx_max]], [p_cb_dist[idx_max]], color='red', s=100, zorder=5, label=f'p_max = {abs(p_cb_dist[idx_max]):.1f} kPa')
axes[1,1].set_xlabel('Distancia radial desde centro (m)', fontweight='bold')
axes[1,1].set_ylabel('Presión (kPa)', fontweight='bold')
axes[1,1].set_title('Presión en Fondo — Convectivo', fontweight='bold')
axes[1,1].grid(True, alpha=0.3, linestyle=':')
axes[1,1].legend(loc='best')
axes[1,1].set_xlim([-D_num/2*1.05, D_num/2*1.05])

plt.tight_layout()
plt.show()

## 9.2 Comparación Impulsivo vs. Convectivo

In [ ]:
# Crear figura comparativa
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Comparación Impulsivo vs. Convectivo', fontsize=14, fontweight='bold')

# 1. Comparación de Masas
masas_labels = ['Impulsiva\n(m_i)', 'Convectiva\n(m_c)', 'Estructural\n(M_pared+M_techo)']
masas_values = [float(m_i), float(m_c), float(M_pared + M_techo)]
colors_masa = ['#2E86AB', '#A23B72', '#F18F01']

bars1 = axes[0,0].bar(masas_labels, masas_values, color=colors_masa, edgecolor='black', linewidth=1.5)
axes[0,0].set_ylabel('Masa (toneladas)', fontweight='bold')
axes[0,0].set_title('Distribución de Masas', fontweight='bold')
axes[0,0].grid(True, axis='y', alpha=0.3, linestyle=':')
for i, (bar, val) in enumerate(zip(bars1, masas_values)):
    height = bar.get_height()
    axes[0,0].text(bar.get_x() + bar.get_width()/2., height,
                   f'{val:.0f} ton\n({val/(float(m_i)+float(m_c))*100:.1f}%)',
                   ha='center', va='bottom', fontsize=9, fontweight='bold')

# 2. Comparación de Periodos
periodos_labels = ['Impulsivo\n(T_i)', 'Convectivo\n(T_c)']
periodos_values = [float(T_i), float(T_c)]
colors_periodo = ['#2E86AB', '#A23B72']

bars2 = axes[0,1].bar(periodos_labels, periodos_values, color=colors_periodo, edgecolor='black', linewidth=1.5)
axes[0,1].set_ylabel('Periodo (s)', fontweight='bold')
axes[0,1].set_title('Periodos Fundamentales', fontweight='bold')
axes[0,1].grid(True, axis='y', alpha=0.3, linestyle=':')
for bar, val in zip(bars2, periodos_values):
    height = bar.get_height()
    axes[0,1].text(bar.get_x() + bar.get_width()/2., height,
                   f'{val:.3f} s',
                   ha='center', va='bottom', fontsize=10, fontweight='bold')

# 3. Comparación de Corte Basal
corte_labels = ['Impulsivo\n(V_i)', 'Convectivo\n(V_c)', 'SRSS\n(V)']
corte_values = [float(V_i), float(V_c), float(V)]
colors_corte = ['#2E86AB', '#A23B72', '#C73E1D']

bars3 = axes[1,0].bar(corte_labels, corte_values, color=colors_corte, edgecolor='black', linewidth=1.5)
axes[1,0].set_ylabel('Corte Basal (kN)', fontweight='bold')
axes[1,0].set_title('Corte Basal Sísmico', fontweight='bold')
axes[1,0].grid(True, axis='y', alpha=0.3, linestyle=':')
for bar, val in zip(bars3, corte_values):
    height = bar.get_height()
    axes[1,0].text(bar.get_x() + bar.get_width()/2., height,
                   f'{val:.0f} kN',
                   ha='center', va='bottom', fontsize=9, fontweight='bold')

# 4. Comparación de Momento Volcante
momento_labels = ['Impulsivo\n(M\'_i)', 'Convectivo\n(M\'_c)', 'SRSS\n(M\')']
momento_values = [float(M_prime_i), float(M_prime_c), float(M_prime)]
colors_momento = ['#2E86AB', '#A23B72', '#C73E1D']

bars4 = axes[1,1].bar(momento_labels, momento_values, color=colors_momento, edgecolor='black', linewidth=1.5)
axes[1,1].set_ylabel('Momento Volcante (kN·m)', fontweight='bold')
axes[1,1].set_title('Momento Volcante Sísmico', fontweight='bold')
axes[1,1].grid(True, axis='y', alpha=0.3, linestyle=':')
for bar, val in zip(bars4, momento_values):
    height = bar.get_height()
    axes[1,1].text(bar.get_x() + bar.get_width()/2., height,
                   f'{val/1000:.0f}\nkN·m',
                   ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

## 9.3 Esquema del Tanque con Parámetros Clave

In [ ]:
# Crear diagrama esquemático del tanque
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7))
fig.suptitle('Esquema del Tanque Cilíndrico con Parámetros de Diseño', 
             fontsize=14, fontweight='bold')

# ===== VISTA ALZADO (Izquierda) =====
r_val = float(r_tanque)
h_pared_val = float(h_pared)
h_liq_val = float(h_liq)
h_i_val = float(h_i)
h_c_val = float(h_c)
d_max_val = float(d_max)

# Tanque
wall_width = 0.3
ax1.plot([0, 0], [0, h_pared_val], 'k-', linewidth=4, label='Pared')
ax1.plot([2*r_val, 2*r_val], [0, h_pared_val], 'k-', linewidth=4)
ax1.plot([0, 2*r_val], [0, 0], 'k-', linewidth=5, label='Fondo')  # Fondo
ax1.plot([0, 2*r_val], [h_pared_val, h_pared_val], 'k-', linewidth=3, label='Techo')

# Nivel de líquido
ax1.fill_between([0, 2*r_val], 0, h_liq_val, alpha=0.3, color='cyan', label='Líquido')
ax1.plot([0, 2*r_val], [h_liq_val, h_liq_val], 'b--', linewidth=2, label='Nivel líquido')

# Altura de ola
wave_y = h_liq_val + d_max_val
x_wave = np.linspace(0, 2*r_val, 50)
y_wave = h_liq_val + d_max_val * np.sin(2*np.pi*x_wave/(2*r_val))
ax1.plot(x_wave, y_wave, 'r-', linewidth=2, label='Ola sísmica')
ax1.plot([2*r_val+0.5, 2*r_val+0.5], [h_liq_val, h_liq_val+d_max_val], 'r-', linewidth=2)
ax1.text(2*r_val+1, h_liq_val+d_max_val/2, f'd_max\n{d_max_val:.2f} m', 
         fontsize=9, va='center', color='red', fontweight='bold')

# Marcadores de altura
ax1.plot([-1, -0.2], [0, 0], 'k-', linewidth=1)
ax1.plot([-1, -0.2], [h_liq_val, h_liq_val], 'k-', linewidth=1)
ax1.plot([-0.6, -0.6], [0, h_liq_val], 'k-', linewidth=2)
ax1.text(-1.5, h_liq_val/2, f'h_liq\n{h_liq_val:.2f} m', 
         fontsize=10, va='center', fontweight='bold')

# Altura total
ax1.plot([2*r_val+2.5, 2*r_val+2.8], [0, 0], 'k-', linewidth=1)
ax1.plot([2*r_val+2.5, 2*r_val+2.8], [h_pared_val, h_pared_val], 'k-', linewidth=1)
ax1.plot([2*r_val+2.65, 2*r_val+2.65], [0, h_pared_val], 'k-', linewidth=2)
ax1.text(2*r_val+3.5, h_pared_val/2, f'h_total\n{h_pared_val:.2f} m',
         fontsize=10, va='center', fontweight='bold')

# Alturas de masas
# Impulsiva
ax1.plot([r_val-1, r_val+1], [h_i_val, h_i_val], 'b-', linewidth=3, alpha=0.7)
ax1.scatter([r_val], [h_i_val], s=300, color='blue', marker='o', 
            edgecolor='darkblue', linewidth=2, zorder=5, label=f'h_i = {h_i_val:.2f} m')
ax1.text(r_val+1.5, h_i_val, f'h_i (impulsiva)', fontsize=9, va='center', 
         color='blue', fontweight='bold')

# Convectiva
ax1.plot([r_val-1, r_val+1], [h_c_val, h_c_val], 'g-', linewidth=3, alpha=0.7)
ax1.scatter([r_val], [h_c_val], s=300, color='green', marker='s',
            edgecolor='darkgreen', linewidth=2, zorder=5, label=f'h_c = {h_c_val:.2f} m')
ax1.text(r_val+1.5, h_c_val, f'h_c (convectiva)', fontsize=9, va='center',
         color='green', fontweight='bold')

# Diámetro
ax1.plot([0, 2*r_val], [-1.5, -1.5], 'k-', linewidth=2)
ax1.plot([0, 0], [-1.8, -1.2], 'k-', linewidth=1)
ax1.plot([2*r_val, 2*r_val], [-1.8, -1.2], 'k-', linewidth=1)
ax1.text(r_val, -2.5, f'D = {2*r_val:.1f} m', fontsize=11, ha='center', fontweight='bold')

ax1.set_xlim([-3, 2*r_val+5])
ax1.set_ylim([-3, h_pared_val+2])
ax1.set_xlabel('', fontweight='bold')
ax1.set_ylabel('Altura (m)', fontweight='bold')
ax1.set_title('Vista en Alzado', fontweight='bold', fontsize=12)
ax1.grid(True, alpha=0.2, linestyle=':')
ax1.legend(loc='upper left', fontsize=9)
ax1.set_xticks([])

# ===== VISTA EN PLANTA (Derecha) =====
circle = plt.Circle((0, 0), r_val, fill=False, edgecolor='black', linewidth=3)
ax2.add_patch(circle)

# Marcar radio
ax2.plot([0, r_val], [0, 0], 'k-', linewidth=2)
ax2.scatter([0], [0], s=100, color='black', marker='x', linewidth=3)
ax2.text(r_val/2, -0.5, f'r = {r_val:.1f} m', fontsize=11, ha='center', fontweight='bold')

# Indicar espesor de pared
e_pared_val = float(e_pared)
ax2.text(r_val+1, r_val/2, f'e_pared = {e_pared_val*1000:.0f} mm', 
         fontsize=10, ha='left', fontweight='bold',
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

# Datos adicionales
info_text = f"GEOMETRÍA:\n"
info_text += f"• Diámetro: {2*r_val:.1f} m\n"
info_text += f"• Altura total: {h_pared_val:.2f} m\n"
info_text += f"• Altura líquido: {h_liq_val:.2f} m\n"
info_text += f"• Relación h/D: {h_liq_val/(2*r_val):.3f}\n\n"
info_text += f"MASAS:\n"
info_text += f"• Impulsiva: {float(m_i):.0f} ton\n"
info_text += f"• Convectiva: {float(m_c):.0f} ton\n"
info_text += f"• Estructural: {float(M_pared+M_techo):.0f} ton"

ax2.text(-r_val-7, r_val/2, info_text, fontsize=9,
         verticalalignment='center',
         bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.6))

info_text2 = f"PERIODOS:\n"
info_text2 += f"• T_i: {float(T_i):.3f} s\n"
info_text2 += f"• T_c: {float(T_c):.3f} s\n\n"
info_text2 += f"ACELERACIONES:\n"
info_text2 += f"• S_ai: {float(S_ai):.3f} g\n"
info_text2 += f"• S_ac: {float(S_ac):.3f} g"

ax2.text(-r_val-7, -r_val/2, info_text2, fontsize=9,
         verticalalignment='center',
         bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.6))

ax2.set_xlim([-r_val-8, r_val+3])
ax2.set_ylim([-r_val-2, r_val+2])
ax2.set_xlabel('', fontweight='bold')
ax2.set_ylabel('', fontweight='bold')
ax2.set_title('Vista en Planta', fontweight='bold', fontsize=12)
ax2.grid(True, alpha=0.2, linestyle=':')
ax2.set_xticks([])
ax2.set_yticks([])

plt.tight_layout()
plt.show()

---
# 10. DASHBOARD EJECUTIVO DE RESULTADOS

In [ ]:
# Crear tabla resumen ejecutiva
print("="*90)
print(" "*25 + "RESUMEN EJECUTIVO DE RESULTADOS")
print("="*90)
print("\n📊 PROYECTO: Estanque Cilíndrico de Acero")
print("📋 NORMATIVAS: API650 + NCh2369:2025")
print("\n" + "="*90)

print("\n" + "─"*90)
print("1. GEOMETRÍA Y PROPIEDADES")
print("─"*90)
print(f"   • Diámetro (D):              {2*float(r_tanque):>10.2f} m")
print(f"   • Altura total (h_total):    {float(h_pared):>10.2f} m")
print(f"   • Altura líquido (h_liq):    {float(h_liq):>10.2f} m")
print(f"   • Relación h/D:              {float(h_liq)/(2*float(r_tanque)):>10.3f}")
print(f"   • Espesor pared (e_pared):   {float(e_pared)*1000:>10.1f} mm")
print(f"   • Espesor fondo (e_fondo):   {float(e_fondo)*1000:>10.1f} mm")
print(f"   • Volumen líquido:           {float(V_liq):>10.1f} m³")

print("\n" + "─"*90)
print("2. MASAS Y DISTRIBUCIÓN")
print("─"*90)
print(f"   • Masa líquido total:        {float(M_liq):>10.1f} ton")
print(f"       - Impulsiva (m_i):       {float(m_i):>10.1f} ton  ({float(m_i)/float(M_liq)*100:>5.1f}%)")
print(f"       - Convectiva (m_c):      {float(m_c):>10.1f} ton  ({float(m_c)/float(M_liq)*100:>5.1f}%)")
print(f"   • Masa pared:                {float(M_pared):>10.1f} ton")
print(f"   • Masa fondo:                {float(M_fondo):>10.1f} ton")
print(f"   • Masa techo:                {float(M_techo):>10.1f} ton")
print(f"   • Masa total:                {float(M_liq+M_pared+M_fondo+M_techo):>10.1f} ton")

print("\n" + "─"*90)
print("3. ALTURAS DE APLICACIÓN DE FUERZAS")
print("─"*90)
print(f"   • Altura impulsiva (h_i):    {float(h_i):>10.3f} m  (momento base)")
print(f"   • Altura convectiva (h_c):   {float(h_c):>10.3f} m  (momento base)")
print(f"   • Altura impulsiva (h'_i):   {float(h_prime_i):>10.3f} m  (momento volcante)")
print(f"   • Altura convectiva (h'_c):  {float(h_prime_c):>10.3f} m  (momento volcante)")

print("\n" + "─"*90)
print("4. PARÁMETROS SÍSMICOS")
print("─"*90)
print(f"   • Aceleración de referencia: {float(A_r)/float(g):>10.3f} g")
print(f"   • Factor importancia (I):    {float(I):>10.1f}")
print(f"   • Factor de suelo (S):       {float(S):>10.1f}")
print(f"   • R impulsivo (R_i):         {float(R_i):>10.1f}")
print(f"   • R convectivo (R_c):        {float(R_c):>10.1f}")
print(f"   • Amortiguamiento ξ_i:       {float(xi_i)*100:>10.1f} %")
print(f"   • Amortiguamiento ξ_c:       {float(xi_c)*100:>10.1f} %")

print("\n" + "─"*90)
print("5. PERIODOS FUNDAMENTALES")
print("─"*90)
print(f"   • Periodo impulsivo (T_i):   {float(T_i):>10.3f} s")
print(f"   • Periodo convectivo (T_c):  {float(T_c):>10.3f} s")

print("\n" + "─"*90)
print("6. ACELERACIONES ESPECTRALES")
print("─"*90)
print(f"   • S_ai (impulsiva):          {float(S_ai):>10.3f} g")
print(f"   • S_ac (convectiva):         {float(S_ac):>10.3f} g")
print(f"   • S_av (vertical):           {float(S_av):>10.3f} g")

print("\n" + "─"*90)
print("7. FUERZAS SÍSMICAS")
print("─"*90)
print(f"   CORTE BASAL:")
print(f"   • V_i (impulsivo):           {float(V_i):>10.1f} kN")
print(f"   • V_c (convectivo):          {float(V_c):>10.1f} kN")
print(f"   • V (SRSS):                  {float(V):>10.1f} kN  ⚡")
print(f"   • Coef. sísmico equiv.:      {float(V)/(float(m_i+m_c)*float(g)):>10.3f} g")

print("\n" + "─"*90)
print("8. MOMENTOS SÍSMICOS")
print("─"*90)
print(f"   MOMENTO BASE (fondo muro):")
print(f"   • M_i (impulsivo):           {float(M_i):>10.0f} kN·m")
print(f"   • M_c (convectivo):          {float(M_c):>10.0f} kN·m")
print(f"   • M (SRSS):                  {float(M):>10.0f} kN·m  ⚡")
print(f"")
print(f"   MOMENTO VOLCANTE:")
print(f"   • M'_i (impulsivo):          {float(M_prime_i):>10.0f} kN·m")
print(f"   • M'_c (convectivo):         {float(M_prime_c):>10.0f} kN·m")
print(f"   • M' (SRSS):                 {float(M_prime):>10.0f} kN·m  ⚡")

print("\n" + "─"*90)
print("9. PRESIONES HIDRODINÁMICAS MÁXIMAS")
print("─"*90)
print(f"   • p_iw (muro impulsivo):     {float(p_iw):>10.2f} kPa")
print(f"   • p_cw (muro convectivo):    {float(p_cw):>10.2f} kPa")
print(f"   • p_ib (fondo impulsivo):    {float(p_ib):>10.2f} kPa")
print(f"   • p_cb (fondo convectivo):   {float(p_cb):>10.2f} kPa")
print(f"   • p_ww (inercia muro):       {float(p_ww):>10.2f} kPa  (despreciable)")
print(f"   • p_v (sísmica vertical):    {float(p_v):>10.2f} kPa")
print(f"   • p_max (SRSS combinada):    {float(p):>10.2f} kPa  ⚡")
print(f"   • p_estática (referencia):   {float(p_estatico):>10.2f} kPa")
print(f"   • Ratio p_max/p_estática:    {float(rel_presion):>10.3f}")

print("\n" + "─"*90)
print("10. CARGAS LINEALES EQUIVALENTES (para análisis FEA)")
print("─"*90)
print(f"   IMPULSIVO:")
print(f"   • q_i:                       {float(q_i):>10.2f} kN/m")
print(f"   • a_i:                       {float(a_i):>10.4f} kN/m³")
print(f"   • b_i:                       {float(b_i):>10.2f} kN/m²")
print(f"   CONVECTIVO:")
print(f"   • q_c:                       {float(q_c):>10.2f} kN/m")
print(f"   • a_c:                       {float(a_c):>10.4f} kN/m³")
print(f"   • b_c:                       {float(b_c):>10.2f} kN/m²")

print("\n" + "─"*90)
print("11. ALTURA DE OLA Y ANCLAJE")
print("─"*90)
print(f"   • Altura máxima ola (d_max): {float(d_max):>10.3f} m")
print(f"   • Freeboard disponible:      {float(h_pared)-float(h_liq):>10.3f} m")
print(f"   • ¿Requiere anclaje?:        {check_anclaje:>15}")
if "NO" in check_anclaje:
    print(f"       (h/D = {h_liq_num/D_num:.3f} < 1/S_ai = {1/float(S_ai):.3f})")
else:
    print(f"       (h/D = {h_liq_num/D_num:.3f} ≥ 1/S_ai = {1/float(S_ai):.3f})")

print("\n" + "="*90)
print("12. CHECKS DE DISEÑO")
print("="*90)

# Check 1: Freeboard
freeboard = float(h_pared) - float(h_liq)
freeboard_ok = "✓ OK" if freeboard > float(d_max) else "✗ NO CUMPLE"
print(f"   ✏ FREEBOARD:")
print(f"      Disponible: {freeboard:.3f} m  |  Requerido: {float(d_max):.3f} m  →  {freeboard_ok}")

# Check 2: Presión máxima vs estática
ratio_presion = float(p) / float(p_estatico)
presion_ok = "✓ OK" if ratio_presion < 2.0 else "⚠ REVISAR" if ratio_presion < 2.5 else "✗ CRÍTICO"
print(f"\n   ✏ PRESIÓN HIDRODINÁMICA:")
print(f"      Ratio p_max/p_estática: {ratio_presion:.3f}  →  {presion_ok}")

# Check 3: Esbeltez
esbeltez = float(h_liq) / (2*float(r_tanque))
esbeltez_tipo = "Tanque ancho" if esbeltez < 0.5 else "Tanque intermedio" if esbeltez < 1.5 else "Tanque esbelto"
print(f"\n   ✏ ESBELTEZ:")
print(f"      h/D = {esbeltez:.3f}  →  {esbeltez_tipo}")

# Check 4: Periodos
periodo_ok_i = "✓ OK" if 0.1 < float(T_i) < 1.0 else "⚠ REVISAR"
periodo_ok_c = "✓ OK" if 1.0 < float(T_c) < 6.0 else "⚠ REVISAR"
print(f"\n   ✏ PERIODOS (Rangos típicos):")
print(f"      T_i = {float(T_i):.3f} s (típico: 0.1-1.0 s)  →  {periodo_ok_i}")
print(f"      T_c = {float(T_c):.3f} s (típico: 1.0-6.0 s)  →  {periodo_ok_c}")

# Check 5: Anclaje
anclaje_status = "✓ NO REQUIERE" if "NO" in check_anclaje else "⚠ REQUIERE ANCLAJE"
print(f"\n   ✏ ANCLAJE:")
print(f"      Criterio NCh2369: {anclaje_status}")

print("\n" + "="*90)
print(" "*30 + "FIN DEL REPORTE")
print("="*90)

## Tabla de Coeficientes para Análisis Estructural

In [ ]:
# Tabla formateada para uso en FEA
import pandas as pd

# Crear tabla de presiones lineales equivalentes
data_presiones = {
    'Modo': ['Impulsivo', 'Convectivo'],
    'q (kN/m)': [float(q_i), float(q_c)],
    'a (kN/m³)': [float(a_i), float(a_c)],
    'b (kN/m²)': [float(b_i), float(b_c)],
    'h_aplicación (m)': [float(h_i), float(h_c)]
}

df_presiones = pd.DataFrame(data_presiones)
print("\n" + "="*80)
print("COEFICIENTES DE PRESIÓN LINEAL EQUIVALENTE (para modelo FEA)")
print("="*80)
print("\nModelo de presión: p(y) = a·y² + b·y")
print("Carga resultante: q (aplicada en altura h_aplicación)\n")
print(df_presiones.to_string(index=False))
print("\n" + "="*80)

# Crear tabla resumen de resultados clave
data_resumen = {
    'Parámetro': [
        'Diámetro D',
        'Altura líquido h_liq',
        'Relación h/D',
        'Masa impulsiva m_i',
        'Masa convectiva m_c',
        'Periodo impulsivo T_i',
        'Periodo convectivo T_c',
        'Aceleración S_ai',
        'Aceleración S_ac',
        'Corte basal V',
        'Momento base M',
        'Momento volcante M\'',
        'Presión máxima p_max',
        'Altura ola d_max'
    ],
    'Valor': [
        f"{2*float(r_tanque):.2f}",
        f"{float(h_liq):.2f}",
        f"{float(h_liq)/(2*float(r_tanque)):.3f}",
        f"{float(m_i):.1f}",
        f"{float(m_c):.1f}",
        f"{float(T_i):.3f}",
        f"{float(T_c):.3f}",
        f"{float(S_ai):.3f}",
        f"{float(S_ac):.3f}",
        f"{float(V):.0f}",
        f"{float(M):.0f}",
        f"{float(M_prime):.0f}",
        f"{float(p):.2f}",
        f"{float(d_max):.3f}"
    ],
    'Unidad': [
        'm',
        'm',
        '-',
        'ton',
        'ton',
        's',
        's',
        'g',
        'g',
        'kN',
        'kN·m',
        'kN·m',
        'kPa',
        'm'
    ]
}

df_resumen = pd.DataFrame(data_resumen)
print("\n" + "="*80)
print("RESULTADOS CLAVE — RESUMEN COMPACTO")
print("="*80)
print(df_resumen.to_string(index=False))
print("\n" + "="*80)

---
## 📝 Notas Finales

### Criterios de Diseño Aplicados:
- ✅ **API 650**: Geometría y cargas estáticas
- ✅ **NCh2369:2025**: Análisis sísmico (modelo impulsivo-convectivo)
- ✅ **SRSS**: Combinación de modos mediante raíz cuadrada de suma de cuadrados

### Próximos Pasos:
1. Verificar esfuerzos en pared (tensión circunferencial, flexión)
2. Diseñar anclaje si es requerido
3. Revisar uniones fondo-pared y pared-techo
4. Validar freeboard considerando ola sísmica
5. Análisis de estabilidad al volcamiento

### Referencias:
- API Standard 650, Welded Tanks for Oil Storage (12th Edition)
- NCh2369:2025, Diseño sísmico de estructuras e instalaciones industriales
- Housner, G.W. (1963), "The dynamic behavior of water tanks"